# CBIS notebook réorganisé

Notebook remis en ordre pour:
- séparer exploration, préparation, dataloaders, modèle, entraînement;
- éviter de charger toutes les images en RAM d'un coup;
- réduire le temps de calcul;
- rendre chaque cellule identifiable rapidement.


## 1. Imports et configuration

Cette section contient seulement les dépendances et les chemins globaux.


In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.utils.data import WeightedRandomSampler

from tqdm.notebook import tqdm

try:
    import torchvision.transforms as T
except ImportError:
    T = None

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device =', DEVICE)

IMAGES_ROOT = Path('data/CBIS/jpeg')
CSV_PATH = Path('data/CBIS/csv/mass_case_description_train_set.csv')


device = cpu


## 2. Lecture du CSV

On ne touche pas encore aux images. On commence par préparer une table propre avec chemins et labels.


In [2]:
df = pd.read_csv(CSV_PATH)
print(df.shape)
df.head()


(1318, 14)


,patient_id,breast_density,left or right breast,image view,abnormality id,abnormality type,mass shape,mass margins,assessment,pathology,subtlety,image file path,cropped image file path,ROI mask file path
0,P_00001,3,LEFT,CC,1,mass,IRREGULAR-ARCHITECTURAL_DISTORTION,SPICULATED,4,MALIGNANT,4,Mass-Training_P_00001_LEFT_CC/1.3.6.1.4.1.9590...,Mass-Training_P_00001_LEFT_CC_1/1.3.6.1.4.1.95...,Mass-Training_P_00001_LEFT_CC_1/1.3.6.1.4.1.95...
1,P_00001,3,LEFT,MLO,1,mass,IRREGULAR-ARCHITECTURAL_DISTORTION,SPICULATED,4,MALIGNANT,4,Mass-Training_P_00001_LEFT_MLO/1.3.6.1.4.1.959...,Mass-Training_P_00001_LEFT_MLO_1/1.3.6.1.4.1.9...,Mass-Training_P_00001_LEFT_MLO_1/1.3.6.1.4.1.9...
2,P_00004,3,LEFT,CC,1,mass,ARCHITECTURAL_DISTORTION,ILL_DEFINED,4,BENIGN,3,Mass-Training_P_00004_LEFT_CC/1.3.6.1.4.1.9590...,Mass-Training_P_00004_LEFT_CC_1/1.3.6.1.4.1.95...,Mass-Training_P_00004_LEFT_CC_1/1.3.6.1.4.1.95...
3,P_00004,3,LEFT,MLO,1,mass,ARCHITECTURAL_DISTORTION,ILL_DEFINED,4,BENIGN,3,Mass-Training_P_00004_LEFT_MLO/1.3.6.1.4.1.959...,Mass-Training_P_00004_LEFT_MLO_1/1.3.6.1.4.1.9...,Mass-Training_P_00004_LEFT_MLO_1/1.3.6.1.4.1.9...
4,P_00004,3,RIGHT,MLO,1,mass,OVAL,CIRCUMSCRIBED,4,BENIGN,5,Mass-Training_P_00004_RIGHT_MLO/1.3.6.1.4.1.95...,Mass-Training_P_00004_RIGHT_MLO_1/1.3.6.1.4.1....,Mass-Training_P_00004_RIGHT_MLO_1/1.3.6.1.4.1....


In [3]:
PATH_COL = 'image file path'
LABEL_COL = 'pathology'

assert PATH_COL in df.columns, f"Colonne manquante: {PATH_COL}"
assert LABEL_COL in df.columns, f"Colonne manquante: {LABEL_COL}"

print(df[[PATH_COL, LABEL_COL]].head())


                                     image file path  pathology
0  Mass-Training_P_00001_LEFT_CC/1.3.6.1.4.1.9590...  MALIGNANT
1  Mass-Training_P_00001_LEFT_MLO/1.3.6.1.4.1.959...  MALIGNANT
2  Mass-Training_P_00004_LEFT_CC/1.3.6.1.4.1.9590...     BENIGN
3  Mass-Training_P_00004_LEFT_MLO/1.3.6.1.4.1.959...     BENIGN
4  Mass-Training_P_00004_RIGHT_MLO/1.3.6.1.4.1.95...     BENIGN


## 3. Construire une table légère

Au lieu d'ouvrir les images dans `get_data()`, on stocke seulement:
- le chemin du fichier `.jpg`;
- le label binaire.

C'est le point principal qui accélère le notebook.


In [4]:
df = pd.read_csv(CSV_PATH).copy()

df['label'] = df['pathology'].map({
    'BENIGN': 0,
    'BENIGN_WITHOUT_CALLBACK': 0,
    'MALIGNANT': 1
})

df = df.dropna(subset=['label']).copy()
df['label'] = df['label'].astype(int)

def csv_path_to_jpg_folder(csv_path_str):
    parts = str(csv_path_str).split('/')
    if len(parts) < 3:
        return None
    return IMAGES_ROOT / parts[2]

def first_jpg_in_folder(folder):
    if folder is None or not Path(folder).exists():
        return None
    files = sorted([f for f in os.listdir(folder) if f.lower().endswith('.jpg')])
    if not files:
        return None
    return str(Path(folder) / files[0])

# on prend de préférence le chemin "image file path"
df['image_folder'] = df['image file path'].apply(csv_path_to_jpg_folder)
df['image_path'] = df['image_folder'].apply(first_jpg_in_folder)

meta_df = df.dropna(subset=['image_path']).copy()

print(meta_df.shape)
display(meta_df[['image file path', 'image_path', 'label']].head())
print(meta_df['label'].value_counts())

(1318, 17)


,image file path,image_path,label
0,Mass-Training_P_00001_LEFT_CC/1.3.6.1.4.1.9590...,data\CBIS\jpeg\1.3.6.1.4.1.9590.100.1.2.342386...,1
1,Mass-Training_P_00001_LEFT_MLO/1.3.6.1.4.1.959...,data\CBIS\jpeg\1.3.6.1.4.1.9590.100.1.2.359308...,1
2,Mass-Training_P_00004_LEFT_CC/1.3.6.1.4.1.9590...,data\CBIS\jpeg\1.3.6.1.4.1.9590.100.1.2.891800...,0
3,Mass-Training_P_00004_LEFT_MLO/1.3.6.1.4.1.959...,data\CBIS\jpeg\1.3.6.1.4.1.9590.100.1.2.295360...,0
4,Mass-Training_P_00004_RIGHT_MLO/1.3.6.1.4.1.95...,data\CBIS\jpeg\1.3.6.1.4.1.9590.100.1.2.410524...,0


label
0    681
1    637
Name: count, dtype: int64


In [5]:
meta_df['label'].value_counts()


label
0    681
1    637
Name: count, dtype: int64

## 4. Split train / validation rapide

On garde un petit sous-ensemble pour tester vite, puis on pourra agrandir plus tard.


In [6]:
def stratified_split(df, label_col='label', train_frac=0.8, seed=SEED):
    train_parts = []
    val_parts = []
    for label, group in df.groupby(label_col):
        group = group.sample(frac=1.0, random_state=seed).reset_index(drop=True)
        cut = int(len(group) * train_frac)
        train_parts.append(group.iloc[:cut])
        val_parts.append(group.iloc[cut:])
    train_df = pd.concat(train_parts).sample(frac=1.0, random_state=seed).reset_index(drop=True)
    val_df = pd.concat(val_parts).sample(frac=1.0, random_state=seed).reset_index(drop=True)
    return train_df, val_df

train_df, val_df = stratified_split(meta_df)
print('train:', train_df.shape, 'val:', val_df.shape)


train: (1053, 17) val: (265, 17)


In [7]:
FAST_MODE = True
FAST_TRAIN_SAMPLES = 512
FAST_VAL_SAMPLES = 128

if FAST_MODE:
    train_df = train_df.groupby('label', group_keys=False).head(FAST_TRAIN_SAMPLES // 2).reset_index(drop=True)
    val_df = val_df.groupby('label', group_keys=False).head(FAST_VAL_SAMPLES // 2).reset_index(drop=True)

print('train used:', train_df.shape)
print('val used:', val_df.shape)


train used: (512, 17)
val used: (128, 17)


## 5. Transformations

On réduit la taille d'image pour accélérer.
`128x128` est bien plus léger que `224x224` pour des essais.


In [ ]:
if T is not None:
    train_transform = T.Compose([
        T.Resize((128, 128)),
        T.Grayscale(num_output_channels=1),
        T.ToTensor(),
    ])
    val_transform = T.Compose([
        T.Resize((128, 128)),
        T.Grayscale(num_output_channels=1),
        T.ToTensor(),
    ])
else:
    train_transform = None
    val_transform = None


## 6. Dataset propre

Les images sont ouvertes à la volée dans `__getitem__`.
Donc:
- pas de `print(get_data()[0])` sur des milliers d'images;
- pas de préchargement complet en mémoire;
- DataLoader plus standard.


In [9]:
class CBISDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.dataframe = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        row = self.dataframe.iloc[idx]
        image = Image.open(row['image_path']).convert('RGB')

        if self.transform is not None:
            image = self.transform(image)
        else:
            image = image.resize((128, 128))
            image = torch.from_numpy(np.array(image)).permute(2, 0, 1).float() / 255.0

        label = torch.tensor(row['label'], dtype=torch.float32)
        return image, label


## 7. DataLoaders


In [ ]:
BATCH_SIZE = 32
NUM_WORKERS = 0

# Dataset
train_dataset = CBISDataset(train_df, transform=train_transform)
val_dataset = CBISDataset(val_df, transform=val_transform)

# Comptage des classes dans le train set
class_counts = train_df['label'].value_counts().sort_index()
print("Class counts:")
print(class_counts)

# Poids inverses par classe
class_weights = 1.0 / class_counts
print("\nClass weights:")
print(class_weights)

# Poids par échantillon
sample_weights = train_df['label'].map(class_weights).values
sample_weights = torch.DoubleTensor(sample_weights)

# Sampler équilibré
train_sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True
)

# DataLoaders
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    sampler=train_sampler,
    num_workers=0
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=0
)

len(train_dataset), len(val_dataset)


(512, 128)

In [11]:
images, labels = next(iter(train_loader))
print(images.shape, labels.shape)


torch.Size([32, 3, 128, 128]) torch.Size([32])


## 8. Modèle simple

Le flatten dépend maintenant de `128x128`.
Après deux max-pool, on obtient `32 x 32 x 32`.


In [ ]:
class Model(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 32 * 32, 128),
            nn.ReLU(),
            nn.Linear(128, 1)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x.squeeze(1)

n_benign = (train_labels == 0).sum()
n_malignant = (train_labels == 1).sum()

pos_weight = torch.tensor([n_benign / n_malignant], device=DEVICE)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
model = Model().to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
model


Model(
  (features): Sequential(
    (0): Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=32768, out_features=128, bias=True)
    (2): ReLU()
    (3): Linear(in_features=128, out_features=1, bias=True)
  )
)

## 9. Boucles train / eval


In [13]:
def run_epoch(model, loader, criterion, optimizer=None, device=DEVICE, desc="Train"):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    batch_bar = tqdm(loader, desc=desc, unit="batch", leave=False)

    for images, labels in batch_bar:
        images = images.to(device)
        labels = labels.to(device)

        with torch.set_grad_enabled(is_train):
            logits = model(images)
            loss = criterion(logits, labels)
            probs = torch.sigmoid(logits)
            preds = (probs >= 0.5).float()

            if is_train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

        batch_size = labels.size(0)
        total_loss += loss.item() * batch_size
        total_correct += (preds == labels).sum().item()
        total_samples += batch_size

        batch_bar.set_postfix(
            loss=f"{total_loss / total_samples:.4f}",
            acc=f"{total_correct / total_samples:.4f}"
        )

    return total_loss / total_samples, total_correct / total_samples

In [14]:
EPOCHS = 5
history = []

for epoch in range(EPOCHS):
    train_loss, train_acc = run_epoch(
        model, train_loader, criterion, optimizer=optimizer, desc=f"Train {epoch+1}/{EPOCHS}"
    )
    val_loss, val_acc = run_epoch(
        model, val_loader, criterion, optimizer=None, desc=f"Val   {epoch+1}/{EPOCHS}"
    )

    history.append({
        'epoch': epoch + 1,
        'train_loss': train_loss,
        'train_acc': train_acc,
        'val_loss': val_loss,
        'val_acc': val_acc,
    })

    print(
        f"Epoch {epoch+1}/{EPOCHS} | "
        f"train_loss={train_loss:.4f} train_acc={train_acc:.4f} | "
        f"val_loss={val_loss:.4f} val_acc={val_acc:.4f}"
    )

Train 1/5:   0%|          | 0/16 [00:00<?, ?batch/s]

Val   1/5:   0%|          | 0/4 [00:00<?, ?batch/s]

Epoch 1/5 | train_loss=0.7071 train_acc=0.4941 | val_loss=0.6764 val_acc=0.5781


Train 2/5:   0%|          | 0/16 [00:00<?, ?batch/s]

Val   2/5:   0%|          | 0/4 [00:00<?, ?batch/s]

...

Val   4/5:   0%|          | 0/4 [00:00<?, ?batch/s]

Epoch 4/5 | train_loss=0.6458 train_acc=0.6406 | val_loss=0.6290 val_acc=0.6406


Train 5/5:   0%|          | 0/16 [00:00<?, ?batch/s]

Val   5/5:   0%|          | 0/4 [00:00<?, ?batch/s]

Epoch 5/5 | train_loss=0.6114 train_acc=0.6875 | val_loss=0.6100 val_acc=0.6406


## 10. Historique


In [30]:
history_df = pd.DataFrame(history)
history_df


,epoch,train_loss,train_acc,val_loss,val_acc
0,1,0.707132,0.494141,0.676403,0.578125
1,2,0.688800,0.531250,0.671534,0.648438
2,3,0.669633,0.597656,0.651490,0.625000
3,4,0.645843,0.640625,0.628992,0.640625
4,5,0.611423,0.687500,0.610024,0.640625


## 11. À retenir

- **Ne plus utiliser** `get_data()` qui ouvre toutes les images avant le training.
- **Ne plus faire** `print(get_data()[0])`, ça relance tout le chargement et bloque vite.
- Le bon flux est: `CSV -> meta_df -> Dataset -> DataLoader -> train`.
- Pour accélérer encore: diminuer la taille d'image, utiliser un sous-ensemble, puis passer à GPU si disponible.
